# Notebook 03-UNSW — Per-Class Isotonic Calibration

**Project:** Calibrated and Stability-Aware Explainable Intrusion Detection  
**Author:** Md Anas Biswas, University of Portsmouth  
**Contribution:** C1 — Per-class isotonic calibration (validated on NSL-KDD with 47–98% ECE reductions and on CIC-IDS2017 with 47–91% reductions)

## What this notebook does

1. Splits the UNSW-NB15 test set 50/50 (stratified by 5-class label) into:
   - A **calibration half** — used to fit isotonic regressors
   - An **evaluation half** — used to measure ECE before and after
2. For each of the 6 models, fits per-class isotonic regression on the calibration half (one regressor per class, one-vs-rest)
3. Computes calibration metrics **before** and **after** for the evaluation half:
   - ECE-15 (15 equal-width bins)
   - Adaptive-ECE (equal-mass bins)
   - MCE (maximum calibration error)
   - Brier score
4. Saves calibrators and calibrated probabilities for downstream use (SCTS-v2 in Notebook 11, conformal coverage in Notebook 12 — out of scope for UNSW)
5. Plots reliability diagrams (uncalibrated vs calibrated) per model
6. Commits and pushes

## Why per-class isotonic specifically (from Kull et al., NeurIPS 2019)

Class-marginal calibration fixes the high-frequency classes (Normal, R2L) but can leave U2R catastrophically miscalibrated. Per-class isotonic regression fits one calibrator per class via one-vs-rest, so every class — including U2R — gets independently calibrated. Isotonic is non-parametric, monotonic (preserves rank ordering), and outperformed temperature scaling and Platt scaling on NSL-KDD and CIC-IDS2017 in earlier notebooks.

## 1. Environment setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, shutil
shutil.copy('/content/drive/MyDrive/XIDS_Research/.gitconfig',       '/root/.gitconfig')
shutil.copy('/content/drive/MyDrive/XIDS_Research/.git-credentials', '/root/.git-credentials')

PROJECT_DIR = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(PROJECT_DIR)

!git config --global credential.helper 'store --file /content/drive/MyDrive/XIDS_Research/.git-credentials'
!git pull origin main

## 2. Load data, models, and existing predictions

In [ ]:
import numpy as np
import json
import joblib
from pathlib import Path
from collections import defaultdict

REPO       = Path(PROJECT_DIR)
PROC_DIR   = REPO / 'data/processed/unsw_nb15'
MODELS_DIR = REPO / 'models/unsw_nb15'
PROBS_DIR  = MODELS_DIR / 'probabilities'
CALIB_DIR  = REPO / 'calibrators/unsw_nb15'
CALIB_DIR.mkdir(parents=True, exist_ok=True)
(CALIB_DIR / 'calibrated_probas').mkdir(exist_ok=True)

y_test_binary = np.load(PROC_DIR / 'y_test_binary.npy')
y_test_5class = np.load(PROC_DIR / 'y_test_5class.npy')

FIVE_CLASS_NAMES = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']

MODEL_SPECS = {
    'rf_binary_cw':     {'target': 'binary',     'y_true': y_test_binary, 'n_classes': 2},
    'xgb_binary_cw':    {'target': 'binary',     'y_true': y_test_binary, 'n_classes': 2},
    'dnn_binary_cw':    {'target': 'binary',     'y_true': y_test_binary, 'n_classes': 2},
    'rf_5class_smote':  {'target': 'multiclass', 'y_true': y_test_5class, 'n_classes': 5},
    'xgb_5class_smote': {'target': 'multiclass', 'y_true': y_test_5class, 'n_classes': 5},
    'dnn_5class_smote': {'target': 'multiclass', 'y_true': y_test_5class, 'n_classes': 5},
}

# Load all the uncalibrated probability arrays
probas = {}
for name in MODEL_SPECS:
    p = np.load(PROBS_DIR / f'{name}_proba.npy')
    probas[name] = p
    print(f'{name:20s} probas: {p.shape}')

## 3. Build calibration / evaluation split (stratified 50/50)

We stratify by **5-class label** rather than binary so the calibration set has guaranteed U2R coverage. The same indices are used across all 6 models so calibration results are directly comparable.

In [ ]:
from sklearn.model_selection import train_test_split

n_test = len(y_test_5class)
idx_all = np.arange(n_test)

idx_calib, idx_eval = train_test_split(
    idx_all,
    test_size=0.5,
    stratify=y_test_5class,
    random_state=42,
)
idx_calib = np.sort(idx_calib)
idx_eval  = np.sort(idx_eval)

print(f'Calibration half: {len(idx_calib):,} rows')
print(f'Evaluation half:  {len(idx_eval):,} rows')

print('\n5-class breakdown on calibration half:')
for cid in range(5):
    n = (y_test_5class[idx_calib] == cid).sum()
    print(f'  {FIVE_CLASS_NAMES[cid]:8s}: {n:>6,}')

print('\n5-class breakdown on evaluation half:')
for cid in range(5):
    n = (y_test_5class[idx_eval] == cid).sum()
    print(f'  {FIVE_CLASS_NAMES[cid]:8s}: {n:>6,}')

# Save indices so downstream notebooks use the same split
np.save(CALIB_DIR / 'idx_calib.npy', idx_calib)
np.save(CALIB_DIR / 'idx_eval.npy',  idx_eval)
print(f'\nSaved split indices to {CALIB_DIR}')

## 4. Calibration metrics (ECE-15, Adaptive-ECE, MCE, Brier)

In [ ]:
def expected_calibration_error(confidences, accuracies, n_bins=15):
    """Equal-width-binned ECE. Naeini, Cooper, Hauskrecht (AAAI 2015)."""
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n = len(confidences)
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        if i == n_bins - 1:
            mask = (confidences >= lo) & (confidences <= hi)
        else:
            mask = (confidences >= lo) & (confidences < hi)
        if mask.sum() == 0:
            continue
        bin_conf = confidences[mask].mean()
        bin_acc  = accuracies[mask].mean()
        ece += (mask.sum() / n) * abs(bin_conf - bin_acc)
    return float(ece)

def adaptive_ece(confidences, accuracies, n_bins=15):
    """Equal-mass-binned ECE. Avoids low-population bins on skewed conf distributions."""
    n = len(confidences)
    order = np.argsort(confidences)
    confs_sorted = confidences[order]
    accs_sorted  = accuracies[order]
    edges = np.linspace(0, n, n_bins + 1).astype(int)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        if hi <= lo:
            continue
        bin_conf = confs_sorted[lo:hi].mean()
        bin_acc  = accs_sorted[lo:hi].mean()
        ece += ((hi - lo) / n) * abs(bin_conf - bin_acc)
    return float(ece)

def max_calibration_error(confidences, accuracies, n_bins=15):
    """Maximum gap between confidence and accuracy across bins."""
    bin_edges = np.linspace(0, 1, n_bins + 1)
    mce = 0.0
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        mask = (confidences >= lo) & (confidences <= hi) if i == n_bins - 1 \
               else (confidences >= lo) & (confidences < hi)
        if mask.sum() == 0:
            continue
        bin_conf = confidences[mask].mean()
        bin_acc  = accuracies[mask].mean()
        mce = max(mce, abs(bin_conf - bin_acc))
    return float(mce)

def brier_multiclass(proba, y_true, n_classes):
    """Brier score generalised to multi-class: mean over samples of sum_c (p_c - 1[y=c])^2."""
    y_onehot = np.zeros_like(proba)
    y_onehot[np.arange(len(y_true)), y_true] = 1.0
    return float(np.mean(np.sum((proba - y_onehot) ** 2, axis=1)))

def calibration_metrics(proba, y_true, n_classes, n_bins=15):
    """Compute the full metric set using max-confidence + correctness."""
    confidences = proba.max(axis=1)
    predictions = proba.argmax(axis=1)
    accuracies  = (predictions == y_true).astype(float)
    return {
        'ece_15':       expected_calibration_error(confidences, accuracies, n_bins=n_bins),
        'adaptive_ece': adaptive_ece(confidences, accuracies, n_bins=n_bins),
        'mce':          max_calibration_error(confidences, accuracies, n_bins=n_bins),
        'brier':        brier_multiclass(proba, y_true, n_classes),
    }

print('Calibration metric helpers ready.')

## 5. Fit per-class isotonic calibrators on the calibration half

For each model, we fit `n_classes` isotonic regressors (one-vs-rest), then renormalise the calibrated probabilities so each row sums to 1.

In [ ]:
from sklearn.isotonic import IsotonicRegression

def fit_per_class_isotonic(proba_calib, y_calib, n_classes):
    """One IsotonicRegression per class, fit on (P[c], 1[y=c]) over the calibration set."""
    calibrators = []
    for c in range(n_classes):
        ir = IsotonicRegression(out_of_bounds='clip')
        ir.fit(proba_calib[:, c], (y_calib == c).astype(float))
        calibrators.append(ir)
    return calibrators

def apply_per_class_isotonic(proba, calibrators):
    """Apply each class's calibrator and renormalise rows to sum to 1."""
    out = np.zeros_like(proba)
    for c, ir in enumerate(calibrators):
        out[:, c] = ir.predict(proba[:, c])
    row_sums = out.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    return out / row_sums

ALL_RESULTS = {}
CALIBRATED_PROBAS = {}

for name, spec in MODEL_SPECS.items():
    print(f'\n=== {name} ===')
    proba_full = probas[name]
    y_true_full = spec['y_true']
    n_classes = spec['n_classes']

    proba_calib = proba_full[idx_calib]
    proba_eval  = proba_full[idx_eval]
    y_calib     = y_true_full[idx_calib]
    y_eval      = y_true_full[idx_eval]

    # Fit calibrators on calibration half
    calibrators = fit_per_class_isotonic(proba_calib, y_calib, n_classes)

    # Apply to evaluation half
    proba_eval_cal = apply_per_class_isotonic(proba_eval, calibrators)

    # Metrics before and after on the evaluation half
    before = calibration_metrics(proba_eval,     y_eval, n_classes)
    after  = calibration_metrics(proba_eval_cal, y_eval, n_classes)

    # Compute reductions
    reduction = {k: (before[k] - after[k]) / before[k] * 100 if before[k] > 0 else 0.0
                 for k in before}

    ALL_RESULTS[name] = {
        'target': spec['target'],
        'n_classes': n_classes,
        'before': before,
        'after':  after,
        'reduction_pct': reduction,
    }

    # Save calibrators and the full calibrated probabilities for ALL test points
    # (needed for SCTS-v2 and downstream — though we apply to all, the calibrators were fit only on idx_calib)
    proba_full_cal = apply_per_class_isotonic(proba_full, calibrators)
    CALIBRATED_PROBAS[name] = proba_full_cal
    joblib.dump(calibrators, CALIB_DIR / f'{name}_calibrators.joblib')
    np.save(CALIB_DIR / 'calibrated_probas' / f'{name}_proba_calibrated.npy', proba_full_cal)

    print(f'  ECE-15:       {before["ece_15"]:.4f} → {after["ece_15"]:.4f}  ({reduction["ece_15"]:.1f}% reduction)')
    print(f'  Adaptive-ECE: {before["adaptive_ece"]:.4f} → {after["adaptive_ece"]:.4f}  ({reduction["adaptive_ece"]:.1f}%)')
    print(f'  MCE:          {before["mce"]:.4f} → {after["mce"]:.4f}  ({reduction["mce"]:.1f}%)')
    print(f'  Brier:        {before["brier"]:.4f} → {after["brier"]:.4f}  ({reduction["brier"]:.1f}%)')

# Persist the full results dictionary
with open(CALIB_DIR / 'calibration_metrics.json', 'w') as f:
    json.dump(ALL_RESULTS, f, indent=2, default=float)

print('\nAll calibration metrics saved.')

## 6. Summary comparison table

In [ ]:
import pandas as pd

rows = []
for name, res in ALL_RESULTS.items():
    rows.append({
        'Model':     name,
        'Target':    res['target'],
        'ECE before':  round(res['before']['ece_15'], 4),
        'ECE after':   round(res['after']['ece_15'],  4),
        'ECE Δ%':      round(res['reduction_pct']['ece_15'], 1),
        'Adapt-ECE before': round(res['before']['adaptive_ece'], 4),
        'Adapt-ECE after':  round(res['after']['adaptive_ece'],  4),
        'Brier before':  round(res['before']['brier'], 4),
        'Brier after':   round(res['after']['brier'],  4),
    })

summary = pd.DataFrame(rows)
summary_path = REPO / 'results/tables/unsw_calibration_metrics.csv'
summary.to_csv(summary_path, index=False)

print(summary.to_string(index=False))
print(f'\nSaved: {summary_path}')

## 7. Reliability diagrams (uncalibrated vs calibrated)

A reliability diagram plots confidence (x-axis) against empirical accuracy (y-axis) per confidence bin. The diagonal is perfect calibration. The further a curve is from the diagonal, the worse the calibration.

In [ ]:
import matplotlib.pyplot as plt

def reliability_curve(proba, y_true, n_bins=15):
    """Return per-bin (mean_conf, accuracy, count) tuples for max-confidence + correctness."""
    confidences = proba.max(axis=1)
    predictions = proba.argmax(axis=1)
    accuracies  = (predictions == y_true).astype(float)
    bin_edges = np.linspace(0, 1, n_bins + 1)
    xs, ys, cs = [], [], []
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        mask = (confidences >= lo) & (confidences <= hi) if i == n_bins - 1 \
               else (confidences >= lo) & (confidences < hi)
        if mask.sum() == 0:
            continue
        xs.append(confidences[mask].mean())
        ys.append(accuracies[mask].mean())
        cs.append(int(mask.sum()))
    return np.array(xs), np.array(ys), np.array(cs)


fig, axes = plt.subplots(2, 3, figsize=(15, 10))
model_order = [
    'rf_binary_cw', 'xgb_binary_cw', 'dnn_binary_cw',
    'rf_5class_smote', 'xgb_5class_smote', 'dnn_5class_smote',
]

for ax, name in zip(axes.flat, model_order):
    spec = MODEL_SPECS[name]
    y_eval = spec['y_true'][idx_eval]
    proba_eval     = probas[name][idx_eval]
    proba_eval_cal = CALIBRATED_PROBAS[name][idx_eval]

    xs_un, ys_un, _ = reliability_curve(proba_eval, y_eval)
    xs_ca, ys_ca, _ = reliability_curve(proba_eval_cal, y_eval)

    ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Perfect calibration')
    ax.plot(xs_un, ys_un, 'o-', color='#E63946', label='Uncalibrated', markersize=5)
    ax.plot(xs_ca, ys_ca, 's-', color='#2E86AB', label='Calibrated',   markersize=5)

    res = ALL_RESULTS[name]
    ax.set_title(f"{name}\nECE {res['before']['ece_15']:.3f} → {res['after']['ece_15']:.3f} ({res['reduction_pct']['ece_15']:.0f}%)",
                 fontsize=10)
    ax.set_xlabel('Confidence')
    ax.set_ylabel('Empirical accuracy')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend(loc='lower right', fontsize=8)

plt.suptitle('UNSW-NB15 — Reliability diagrams (uncalibrated vs per-class isotonic)', fontsize=13, y=1.00)
plt.tight_layout()

fig_path = REPO / 'results/figures/unsw_reliability_diagrams.png'
plt.savefig(fig_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

## 8. Commit and push

In [ ]:
os.chdir(PROJECT_DIR)
!git add notebooks/03_unsw_calibration.ipynb \
         results/figures/unsw_reliability_diagrams.png \
         results/tables/unsw_calibration_metrics.csv
!git commit -m "Notebook 03-UNSW: per-class isotonic calibration on all 6 models (C1)"
!git push origin main

---

## What to check

1. **ECE reduction percentages** — should be in the same range as NSL-KDD (47–98%) and CIC-IDS2017 (47–91%). Anything dramatically lower (single-digit reductions) would suggest an implementation issue.
2. **Reliability diagrams** — calibrated (blue) curves should hug the diagonal more closely than uncalibrated (red) curves. The DNN should show the largest visual gap.
3. **All six calibrators saved** to `calibrators/unsw_nb15/`.
4. **Calibrated probabilities saved** to `calibrators/unsw_nb15/calibrated_probas/` for downstream consumption.

## Next step

**Notebook 04-UNSW** — SHAP value computation across all six models. This is the longest stage runtime-wise (KernelExplainer on the DNN takes a while), so we'll use stratified subsampling to keep it tractable while preserving representative coverage of all five classes.